# Semantic Model DAX to Excel Batch (Paginated-Style)

This notebook loops through business-unit and date-window parameter sets, runs a filtered DAX query for each set, and creates one styled Excel workbook per export.

Goal: replace parameterized paginated report subscriptions with a lighter, schedulable batch export pipeline.

## CU and Performance Strategy

- Push business logic into DAX so only the final dataset is retrieved.
- Keep Spark shuffle partitions low for report workloads.
- Use Spark post-processing only when needed.
- Convert to Pandas only for final Excel rendering (Excel writer is Python-side).

In [ ]:
# Run this near the top of the notebook before heavy Spark work.
# In Fabric, %%configure applies Spark session settings for efficiency.
%%configure -f
{
  "conf": {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.shuffle.partitions": "8",
    "spark.sql.execution.arrow.pyspark.enabled": "true"
  }
}

In [ ]:
# Install dependencies once per environment if needed.
# In Fabric, prefer adding these in the Environment for production jobs.
%pip install -q sempy openpyxl

In [ ]:
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path
import time

import pandas as pd
import sempy.fabric as fabric

from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.pagebreak import Break
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

In [ ]:
# =========================
# Parameters (edit these)
# =========================
WORKSPACE_NAME = "<your-workspace-name>"
SEMANTIC_MODEL_NAME = "<your-semantic-model-name>"

# Update these model identifiers to match your semantic model.
BUSINESS_UNIT_TABLE = "Business Unit"
BUSINESS_UNIT_COLUMN = "Business Unit"
DATE_TABLE = "Date"
DATE_COLUMN = "Date"

# One Excel workbook is created for each parameter set.
# Dates use ISO format and are inclusive.
PARAMETER_SETS = [
    {"business_unit": "North", "start_date": "2026-01-01", "end_date": "2026-01-31"},
    {"business_unit": "South", "start_date": "2026-01-01", "end_date": "2026-01-31"},
]

# Keep the %%...%% tokens in place; the batch cell replaces them per parameter set.
DAX_QUERY_TEMPLATE = """
EVALUATE
TOPN(
    1000,
    SUMMARIZECOLUMNS(
        'Date'[Calendar Year],
        'Product'[Category],
        KEEPFILTERS(
            TREATAS({"%%BUSINESS_UNIT%%"}, '%%BUSINESS_UNIT_TABLE%%'[%%BUSINESS_UNIT_COLUMN%%])
        ),
        KEEPFILTERS(
            DATESBETWEEN(
                '%%DATE_TABLE%%'[%%DATE_COLUMN%%],
                DATE(%%START_YEAR%%, %%START_MONTH%%, %%START_DAY%%),
                DATE(%%END_YEAR%%, %%END_MONTH%%, %%END_DAY%%)
            )
        ),
        "Sales", [Total Sales],
        "Margin", [Margin]
    ),
    [Sales], DESC
)
"""

# Optional Spark post-processing for very large datasets (filter, regroup, enrich).
USE_SPARK_POST_PROCESSING = False

# Output settings
REPORT_TITLE = "Sales Performance Export"
SHEET_NAME = "Report"
OUTPUT_FOLDER = "/lakehouse/default/Files/paginated_exports"

# Guardrail applied independently to every parameter set.
MAX_ROWS_FOR_EXCEL = 300000

In [ ]:
def run_dax_query(workspace_name: str, semantic_model_name: str, dax_query: str) -> pd.DataFrame:
    start = time.time()
    df = fabric.evaluate_dax(
        dataset=semantic_model_name,
        dax_string=dax_query,
        workspace=workspace_name
    )
    elapsed = round(time.time() - start, 2)
    print(f"DAX executed in {elapsed}s. Rows: {len(df):,}, Columns: {len(df.columns)}")
    return df


def optional_spark_post_processing(df: pd.DataFrame) -> pd.DataFrame:
    if not USE_SPARK_POST_PROCESSING:
        return df

    sdf = spark.createDataFrame(df)
    sdf = sdf.coalesce(8)
    return sdf.toPandas()


def normalize_excel_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, pd.Timestamp):
        value = value.to_pydatetime()
    elif hasattr(value, "item") and not isinstance(value, (str, bytes)):
        try:
            value = value.item()
        except (TypeError, ValueError):
            pass

    if isinstance(value, datetime) and value.tzinfo is not None:
        value = value.replace(tzinfo=None)
    return value


def excel_number_format(column_name: str, value) -> str:
    normalized_name = " ".join(
        str(column_name).lower().replace("_", " ").replace("%", " percent ").split()
    )
    percent_terms = ("percent", "percentage", "pct", "rate")

    if isinstance(value, bool):
        return "General"
    if isinstance(value, datetime):
        return "mmm d, yyyy h:mm AM/PM"
    if isinstance(value, date):
        return "mmm d, yyyy"
    if isinstance(value, Integral):
        return "#,##0;[Red]-#,##0;-"
    if isinstance(value, Real):
        if any(term in normalized_name.split() for term in percent_terms):
            return "0.0%;[Red]-0.0%;-"
        return "#,##0.00;[Red]-#,##0.00;-"
    return "General"


def apply_paginated_style(ws, df: pd.DataFrame, report_title: str):
    if len(df.columns) == 0:
        raise ValueError("Excel export requires at least one result column.")

    title_row = 1
    metadata_row = 2
    spacer_row = 3
    header_row = 4
    first_data_row = 5
    last_col = len(df.columns)
    last_col_letter = get_column_letter(last_col)
    detail_rows_per_page = 45

    dark_blue = "17365D"
    medium_blue = "2F75B5"
    light_blue = "D9EAF7"
    alternate_fill = "F4F7FA"
    border_color = "B7C9D6"
    text_color = "1F1F1F"
    muted_color = "666666"
    thin = Side(border_style="thin", color=border_color)

    ws.merge_cells(start_row=title_row, start_column=1, end_row=title_row, end_column=last_col)
    title_cell = ws.cell(row=title_row, column=1, value=report_title)
    title_cell.font = Font(name="Aptos Display", size=16, bold=True, color="FFFFFF")
    title_cell.fill = PatternFill(fill_type="solid", fgColor=dark_blue)
    title_cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[title_row].height = 28
    for col_idx in range(2, last_col + 1):
        ws.cell(row=title_row, column=col_idx).fill = PatternFill(fill_type="solid", fgColor=dark_blue)

    generated_at = datetime.now().strftime("%B %d, %Y at %I:%M %p")
    metadata_text = f"Generated {generated_at}  |  Records: {len(df):,}"
    ws.merge_cells(start_row=metadata_row, start_column=1, end_row=metadata_row, end_column=last_col)
    metadata_cell = ws.cell(row=metadata_row, column=1, value=metadata_text)
    metadata_cell.font = Font(name="Aptos", size=9, italic=True, color=muted_color)
    metadata_cell.fill = PatternFill(fill_type="solid", fgColor=light_blue)
    metadata_cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[metadata_row].height = 20
    ws.row_dimensions[spacer_row].height = 7

    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=header_row, column=col_idx, value=str(col_name))
        cell.font = Font(name="Aptos", size=10, bold=True, color="FFFFFF")
        cell.fill = PatternFill(fill_type="solid", fgColor=medium_blue)
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[header_row].height = 30

    for row_idx, row in enumerate(df.itertuples(index=False, name=None), start=first_data_row):
        row_fill = alternate_fill if (row_idx - first_data_row) % 2 else "FFFFFF"
        ws.row_dimensions[row_idx].height = 18
        for col_idx, raw_value in enumerate(row, start=1):
            value = normalize_excel_value(raw_value)
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Font(name="Aptos", size=9, color=text_color)
            cell.fill = PatternFill(fill_type="solid", fgColor=row_fill)
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
            cell.number_format = excel_number_format(str(df.columns[col_idx - 1]), value)

            if isinstance(value, bool):
                horizontal_alignment = "center"
            elif isinstance(value, (Integral, Real, date)):
                horizontal_alignment = "right"
            else:
                horizontal_alignment = "left"
            cell.alignment = Alignment(horizontal=horizontal_alignment, vertical="center")

            if isinstance(value, str) and value.startswith(("=", "+", "-", "@")):
                cell.data_type = "s"

    if df.empty:
        message_row = first_data_row
        ws.merge_cells(start_row=message_row, start_column=1, end_row=message_row, end_column=last_col)
        message_cell = ws.cell(row=message_row, column=1, value="No data returned for this parameter set.")
        message_cell.font = Font(name="Aptos", size=10, italic=True, color=muted_color)
        message_cell.alignment = Alignment(horizontal="center", vertical="center")
        message_cell.fill = PatternFill(fill_type="solid", fgColor=alternate_fill)
        message_cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
        ws.row_dimensions[message_row].height = 24
        last_row = message_row
        ws.auto_filter.ref = f"A{header_row}:{last_col_letter}{header_row}"
    else:
        last_row = header_row + len(df)
        table_ref = f"A{header_row}:{last_col_letter}{last_row}"
        table = Table(displayName="ExportTable", ref=table_ref)
        table.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium2",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=False,
            showColumnStripes=False
        )
        ws.add_table(table)

    for col_idx, col_name in enumerate(df.columns, start=1):
        sample_values = [normalize_excel_value(value) for value in df.iloc[:, col_idx - 1].head(500)]
        max_content = max(
            [len(str(col_name))] + [len(str(value)) if value is not None else 0 for value in sample_values]
        )
        width = min(max(max_content + 2, 11), 42)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.freeze_panes = f"A{first_data_row}"
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90
    ws.auto_filter.ref = f"A{header_row}:{last_col_letter}{max(header_row, last_row)}"
    ws.print_area = f"A1:{last_col_letter}{last_row}"
    ws.print_title_rows = f"1:{header_row}"

    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = ws.ORIENTATION_LANDSCAPE
    ws.page_setup.paperSize = ws.PAPERSIZE_LETTER
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0
    ws.page_setup.firstPageNumber = 1
    ws.page_setup.useFirstPageNumber = True
    ws.print_options.horizontalCentered = True
    ws.print_options.verticalCentered = False

    ws.page_margins.left = 0.25
    ws.page_margins.right = 0.25
    ws.page_margins.top = 0.45
    ws.page_margins.bottom = 0.45
    ws.page_margins.header = 0.2
    ws.page_margins.footer = 0.2

    escaped_title = report_title.replace("&", "&&")
    ws.oddHeader.center.text = f"&B{escaped_title}"
    ws.oddFooter.left.text = "Generated &D &T"
    ws.oddFooter.center.text = "Page &P of &N"
    ws.oddFooter.right.text = "&F"

    if not df.empty:
        for break_row in range(first_data_row + detail_rows_per_page, last_row + 1, detail_rows_per_page):
            ws.row_breaks.append(Break(id=break_row))



def export_to_excel(df: pd.DataFrame, output_path: str, sheet_name: str, report_title: str):
    output_dir = Path(output_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)

    wb = Workbook()
    wb.properties.creator = "Microsoft Fabric"
    wb.properties.title = report_title
    wb.properties.subject = "Paginated-style semantic model export"
    wb.properties.description = f"Generated from a Fabric semantic model with {len(df):,} detail rows."

    ws = wb.active
    ws.title = sheet_name[:31]
    apply_paginated_style(ws, df, report_title)

    wb.save(output_path)
    print(f"Excel exported: {output_path}")

In [ ]:
def escape_dax_string(value: str) -> str:
    return str(value).replace('"', '""')


def escape_dax_table_name(value: str) -> str:
    return str(value).replace("'", "''")


def escape_dax_column_name(value: str) -> str:
    return str(value).replace("]", "]]")


def safe_file_part(value: str) -> str:
    cleaned = "".join(character if character.isalnum() else "_" for character in str(value).strip())
    return cleaned.strip("_") or "unnamed"


def build_dax_query(parameter_set: dict):
    required_keys = {"business_unit", "start_date", "end_date"}
    missing_keys = required_keys - parameter_set.keys()
    if missing_keys:
        raise ValueError(f"Parameter set is missing: {', '.join(sorted(missing_keys))}")

    start_date = datetime.strptime(parameter_set["start_date"], "%Y-%m-%d").date()
    end_date = datetime.strptime(parameter_set["end_date"], "%Y-%m-%d").date()
    if start_date > end_date:
        raise ValueError(f"start_date {start_date} is after end_date {end_date}")

    replacements = {
        "%%BUSINESS_UNIT%%": escape_dax_string(parameter_set["business_unit"]),
        "%%BUSINESS_UNIT_TABLE%%": escape_dax_table_name(BUSINESS_UNIT_TABLE),
        "%%BUSINESS_UNIT_COLUMN%%": escape_dax_column_name(BUSINESS_UNIT_COLUMN),
        "%%DATE_TABLE%%": escape_dax_table_name(DATE_TABLE),
        "%%DATE_COLUMN%%": escape_dax_column_name(DATE_COLUMN),
        "%%START_YEAR%%": str(start_date.year),
        "%%START_MONTH%%": str(start_date.month),
        "%%START_DAY%%": str(start_date.day),
        "%%END_YEAR%%": str(end_date.year),
        "%%END_MONTH%%": str(end_date.month),
        "%%END_DAY%%": str(end_date.day),
    }

    dax_query = DAX_QUERY_TEMPLATE
    for token, replacement in replacements.items():
        dax_query = dax_query.replace(token, replacement)

    return dax_query, start_date, end_date


batch_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_results = []
generated_paths = set()

for parameter_index, parameter_set in enumerate(PARAMETER_SETS, start=1):
    business_unit = parameter_set.get("business_unit", f"parameter_set_{parameter_index}")
    try:
        dax_query, start_date, end_date = build_dax_query(parameter_set)
        output_file = (
            f"{safe_file_part(REPORT_TITLE)}_{safe_file_part(business_unit)}_"
            f"{start_date:%Y%m%d}_{end_date:%Y%m%d}_{batch_timestamp}.xlsx"
        )
        output_path = f"{OUTPUT_FOLDER}/{output_file}"
        if output_path in generated_paths:
            raise ValueError(f"Duplicate output path generated: {output_path}")
        generated_paths.add(output_path)

        parameter_title = f"{REPORT_TITLE} - {business_unit} ({start_date} to {end_date})"
        print(f"[{parameter_index}/{len(PARAMETER_SETS)}] Exporting {parameter_title}")

        result_df = run_dax_query(WORKSPACE_NAME, SEMANTIC_MODEL_NAME, dax_query)
        result_df = optional_spark_post_processing(result_df)

        if len(result_df) > MAX_ROWS_FOR_EXCEL:
            raise ValueError(
                f"Result has {len(result_df):,} rows, exceeding "
                f"MAX_ROWS_FOR_EXCEL={MAX_ROWS_FOR_EXCEL:,}."
            )

        export_to_excel(result_df, output_path, SHEET_NAME, parameter_title)
        export_results.append({
            "business_unit": business_unit,
            "start_date": str(start_date),
            "end_date": str(end_date),
            "rows": len(result_df),
            "status": "Succeeded",
            "output_path": output_path,
            "error": None,
        })
    except Exception as error:
        print(f"[{parameter_index}/{len(PARAMETER_SETS)}] Failed for {business_unit}: {error}")
        export_results.append({
            "business_unit": business_unit,
            "start_date": parameter_set.get("start_date"),
            "end_date": parameter_set.get("end_date"),
            "rows": None,
            "status": "Failed",
            "output_path": None,
            "error": str(error),
        })

export_summary = pd.DataFrame(export_results)
display(export_summary)

failed_exports = export_summary[export_summary["status"] == "Failed"]
if not failed_exports.empty:
    raise RuntimeError(f"{len(failed_exports)} of {len(export_summary)} exports failed. See export_summary above.")

## Operational Notes (Subscription Replacement)

- Add one dictionary to `PARAMETER_SETS` for each business unit and inclusive date window.
- Update the business-unit and date table/column names to match the semantic model.
- Each filename includes the report title, business unit, date window, and batch timestamp.
- Successful exports remain in OneLake if another parameter set fails; the notebook fails after displaying the complete status summary.
- Schedule this notebook with a Fabric pipeline or job scheduler and distribute files through downstream automation.